# Generation Eval Results — 2026-03-25

First run of the generation and embellishment evals. Compares `best_fields` (BM25 only, pre-hybrid) against `rrf_elser` (current production) across 10 benchmark questions.

## Key Findings

- **Relevance is strong across both strategies** (~4.5–4.6) — the model is answering the right question regardless of retrieval method
- **Faithfulness is the weak spot** — `best_fields` scores 1.0 on gameplay questions, meaning the model is going significantly beyond what was retrieved
- **`rrf_elser` does not clearly win on correctness or faithfulness** — retrieval quality alone is not the bottleneck for information accuracy
- **Source material is almost entirely dry** (source_score ~1.1–1.3) — both strategies return structured, stat-heavy snippets with little prose
- **The model consistently adds narrative on top** (response_score ~2.4) — this is the system prompt doing the work, not the retrieval
- **Embellishment is concentrated in lore questions** — Arthas, Sylvanas, Lich King, and Illidan all flagged with delta >= 2

In [1]:
import sys, json
from pathlib import Path
sys.path.append(str(Path('../..').resolve()))

import anthropic
from elasticsearch import Elasticsearch
from config import ANTHROPIC_API_KEY, ES_ENDPOINT, ES_API_KEY, ES_INDEX
from evals.generation.judge import score, score_narrative
from evals.generation.run_eval import search_with_snippets, generate

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
es     = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY)

BENCHMARK          = Path('../../evals/generation/benchmark.json')
DIMENSIONS         = ['correctness', 'faithfulness', 'relevance']
COMPARE_STRATEGIES = ['best_fields', 'rrf_elser']

In [2]:
benchmark  = json.loads(BENCHMARK.read_text())
categories = sorted(set(b['category'] for b in benchmark))

print(f'Total questions: {len(benchmark)}')
for cat in categories:
    print(f'  {cat}: {sum(1 for b in benchmark if b["category"] == cat)}')

Total questions: 10
  gameplay: 2
  item: 2
  lore: 6


## Generation Eval Results

Loaded from `evals/generation/results.json`.

In [3]:
gen_results = json.loads(Path('../../evals/generation/results.json').read_text())

col = 14
print(f"{'Dimension':<15}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print(f'  {"delta":>8}')
print('-' * (15 + (col + 2) * len(COMPARE_STRATEGIES) + 10))

for dim in DIMENSIONS:
    avgs  = {n: sum(r[dim] for r in gen_results[n]) / len(gen_results[n]) for n in COMPARE_STRATEGIES}
    delta = avgs[COMPARE_STRATEGIES[-1]] - avgs[COMPARE_STRATEGIES[0]]
    print(f'{dim:<15}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print(f'  {delta:>+8.2f}')

Dimension           best_fields       rrf_elser     delta
---------------------------------------------------------
correctness                3.30            3.00     -0.30
faithfulness               3.00            3.00     +0.00
relevance                  4.50            4.60     +0.10


In [4]:
for category in categories:
    print(f'\n{category}')
    for dim in DIMENSIONS:
        avgs = []
        print(f'  {dim:<15}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in gen_results[name] if r['category'] == category]
            avg = sum(r[dim] for r in cat_results) / len(cat_results)
            avgs.append(avg)
            print(f'  {avg:>12.2f}', end='')
        print(f'  {avgs[-1] - avgs[0]:>+8.2f}')


gameplay
  correctness              4.00          3.50     -0.50
  faithfulness             1.00          2.50     +1.50
  relevance                5.00          5.00     +0.00

item
  correctness              4.00          3.00     -1.00
  faithfulness             4.50          4.50     +0.00
  relevance                4.50          4.50     +0.00

lore
  correctness              2.83          2.83     +0.00
  faithfulness             3.17          2.67     -0.50
  relevance                4.33          4.50     +0.17


## Embellishment Eval Results

Loaded from `evals/generation/embellishment_results.json`.

In [5]:
emb_results = json.loads(Path('../../evals/generation/embellishment_results.json').read_text())

col = 14
print(f"{'':18}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print()
print('-' * (18 + (col + 2) * len(COMPARE_STRATEGIES)))

for field in ['source_score', 'response_score', 'delta']:
    avgs = {n: sum(r[field] for r in emb_results[n]) / len(emb_results[n]) for n in COMPARE_STRATEGIES}
    print(f'{field:<18}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print()

                       best_fields       rrf_elser
--------------------------------------------------
source_score                  1.10            1.30
response_score                2.40            2.40
delta                         1.30            1.10


In [6]:
for category in categories:
    print(f'\n{category}')
    for field in ['source_score', 'response_score', 'delta']:
        print(f'  {field:<18}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in emb_results[name] if r['category'] == category]
            avg = sum(r[field] for r in cat_results) / len(cat_results)
            print(f'  {avg:>12.2f}', end='')
        print()


gameplay
  source_score              1.00          1.50
  response_score            2.00          2.00
  delta                     1.00          0.50

item
  source_score              1.00          1.00
  response_score            2.00          2.50
  delta                     1.00          1.50

lore
  source_score              1.17          1.33
  response_score            2.67          2.50
  delta                     1.50          1.17


In [7]:
for name in COMPARE_STRATEGIES:
    flagged = [r for r in emb_results[name] if r['delta'] >= 2]
    print(f'{name} — {len(flagged)} flagged')
    for r in flagged:
        print(f"  [{r['category']}] {r['question']}")
        print(f"    source={r['source_score']}  response={r['response_score']}  delta={r['delta']:+d}")
        print(f"    {r['response_reasoning']}")
    print()

best_fields — 3 flagged
  [lore] Who is Arthas Menethil?
    source=1  response=4  delta=+3
    The text employs flowing prose with clear cause-and-effect relationships and a distinct voice that guides the reader through Arthas's tragic arc, though occasional bold topic sentences and references to external sources slightly interrupt the narrative momentum.

  [lore] Who is Sylvanas Windrunner?
    source=1  response=3  delta=+2
    The text uses flowing sentences and connective prose to describe Sylvanas' character arc with cause and effect, but it's still organized around bold topic sentences and maintains a somewhat structured, reference-like tone typical of character summaries.

  [lore] What is the Lich King?
    source=1  response=3  delta=+2
    The text blends structured factual exposition with flowing narrative passages that describe the Lich King's threat and his climactic confrontation, creating a mixed style that is neither purely list-like nor fully immersive prose.

rrf_el